Q1

In [ ]:
import pandas as pd
import nltk
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

# Use the raw tickets generated by the starter notebook.
# If your variable has a different name, change this assignment.
raw_tickets = tickets

stop_words = set(stopwords.words("english"))
punctuation_set = set(string.punctuation)

def get_raw_text(ticket):
    if isinstance(ticket, str):
        return ticket
    if isinstance(ticket, dict):
        for key in ("text", "description", "ticket", "content", "message"):
            if key in ticket and isinstance(ticket[key], str):
                return ticket[key]
    raise TypeError("Could not identify the raw text of this ticket.")

rows = []
ticket_details = []

for i, ticket in enumerate(raw_tickets, start=1):
    raw_text = get_raw_text(ticket)

    # Required call to the assignment's original function
    processed = process_ticket(ticket)

    sentences = sent_tokenize(raw_text)
    tokens = word_tokenize(raw_text)

    punctuation_tokens = [
        token for token in tokens
        if all(char in string.punctuation for char in token)
    ]

    stopword_tokens = [
        token for token in tokens
        if token.lower() in stop_words
    ]

    # Final tokens are obtained from the starter function's output.
    if isinstance(processed, dict):
        final_tokens = None
        for key in ("final_tokens", "final", "cleaned_tokens",
                    "filtered_tokens", "tokens"):
            if key in processed and isinstance(processed[key], (list, tuple)):
                final_tokens = list(processed[key])
                break
        if final_tokens is None:
            raise KeyError(
                "Inspect process_ticket(ticket) and update the final-token key."
            )
    elif isinstance(processed, (list, tuple)):
        final_tokens = list(processed)
    else:
        raise TypeError(
            "Inspect the return type of process_ticket(ticket) "
            "to identify its final tokens."
        )

    # Count tokens excluding punctuation tokens.
    non_punctuation_tokens = [
        token for token in tokens
        if not all(char in string.punctuation for char in token)
    ]

    rows.append({
        "Ticket": i,
        "Sentences": len(sentences),
        "Tokens": len(tokens),
        "Punctuation Tokens": len(punctuation_tokens),
        "Stopwords": len(stopword_tokens),
        "Unique Tokens": len(set(non_punctuation_tokens)),
        "Final Tokens": len(final_tokens)
    })

    ticket_details.append({
        "Ticket": i,
        "Actual Tokens": tokens,
        "Punctuation Tokens": punctuation_tokens,
        "Stopwords": stopword_tokens,
        "Final Tokens": final_tokens
    })

df_preprocessing = pd.DataFrame(rows)

count_columns = [
    "Sentences", "Tokens", "Punctuation Tokens",
    "Stopwords", "Unique Tokens", "Final Tokens"
]

total_row = {"Ticket": "TOTAL"}
for column in count_columns:
    total_row[column] = df_preprocessing[column].sum()

df_preprocessing = pd.concat(
    [df_preprocessing, pd.DataFrame([total_row])],
    ignore_index=True
)

display(df_preprocessing)

# Display the intermediate results for the first ticket
selected_ticket = ticket_details[0]

for key, value in selected_ticket.items():
    print(f"\n{key}:")
    print(value)

Q2

In [ ]:
import re

# Keep the original TEST_LINES supplied by the starter notebook.
test_lines = TEST_LINES

patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers, decimals and versions": r"(?<![\w.])\d+(?:\.\d+)*(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words beginning with a vowel": r"\b[aeiouAEIOU][a-zA-Z]*\b"
}

def extract_patterns(text):
    return {
        name: re.findall(pattern, text)
        for name, pattern in patterns.items()
    }

def replace_sensitive_data(text):
    # URLs first, so their contents are not mistaken for email addresses.
    text = re.sub(
        r"\b(?:https?://|www\.)[^\s<>]+",
        "<URL>",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "<EMAIL>",
        text
    )

    # Supports common formats such as +91-9876543210 and (555) 123-4567.
    text = re.sub(
        r"(?<!\w)(?:\+?\d{1,3}[-.\s]?)?"
        r"(?:\(\d{2,4}\)|\d{2,4})[-.\s]?"
        r"\d{3,4}[-.\s]?\d{4}(?!\w)",
        "<PHONE>",
        text
    )

    return text

# Apply extraction and replacement to every raw ticket.
for i, ticket in enumerate(raw_tickets, start=1):
    text = get_raw_text(ticket)

    print(f"\n{'=' * 20} TICKET {i} {'=' * 20}")
    for category, matches in extract_patterns(text).items():
        print(f"{category}: {matches}")

    print("Original:", text)
    print("Replaced:", replace_sensitive_data(text))

# Test all supplied test lines
for line in test_lines:
    print("\nTEST LINE:", line)
    print("Extracted:", extract_patterns(line))
    print("Replaced:", replace_sensitive_data(line))

Q3

In [ ]:
import re
import nltk
from nltk.tokenize import RegexpTokenizer, word_tokenize

# One RegexpTokenizer pattern.
# Order matters: URLs, emails and numbers must be matched before words.
token_pattern = (
    r"https?://[^\s<>]+|www\.[^\s<>]+"
    r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    r"|(?:\+?\d{1,3}[-.\s]?)?(?:\(\d{2,4}\)|\d{2,4})[-.\s]?\d{3,4}[-.\s]?\d{4}"
    r"|\d+(?:\.\d+)+"
    r"|\d+(?:\.\d+)?"
    r"|[A-Za-z]+(?:-[A-Za-z]+)+"
    r"|[A-Za-z]+(?:'[A-Za-z]+)+"
    r"|[A-Za-z]+"
    r"|[^\w\s]"
)

custom_tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return custom_tokenizer.tokenize(text)

# Test on the supplied test lines
for line in TEST_LINES:
    print("\nText:", line)
    print("Custom tokens:", my_tokenize(line))
    print("NLTK word_tokenize:", word_tokenize(line))

# Test on every generated ticket
for i, ticket in enumerate(raw_tickets, start=1):
    text = get_raw_text(ticket)
    print(f"\nTicket {i}")
    print("Custom:", my_tokenize(text))
    print("NLTK:", word_tokenize(text))

# Show differences for each test line
for line in TEST_LINES:
    custom = my_tokenize(line)
    nltk_tokens = word_tokenize(line)

    if custom != nltk_tokens:
        print("\nDifferent tokenization:")
        print("Text:", line)
        print("Custom tokenizer:", custom)
        print("NLTK tokenizer:", nltk_tokens)

Q4

In [ ]:
import nltk
import pandas as pd
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

# Collect unique final tokens from Question 1
unique_final_tokens = sorted({
    str(token).lower()
    for item in ticket_details
    for token in item["Final Tokens"]
    if str(token).isalpha()
})

porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

# Find suffixes occurring in at least two words in this dataset.
suffix_candidates = ["ing", "ed", "ies", "es", "s", "ly", "tion"]
suffix_counts = {
    suffix: [
        word for word in unique_final_tokens
        if len(word) > len(suffix) and word.endswith(suffix)
    ]
    for suffix in suffix_candidates
}

valid_suffixes = [
    suffix for suffix, words in suffix_counts.items()
    if len(words) >= 2
]

if valid_suffixes:
    chosen_suffix = max(valid_suffixes, key=lambda s: len(suffix_counts[s]))
else:
    chosen_suffix = "ing"
    print("No candidate suffix occurs in two or more final tokens.")

print("Chosen suffix:", chosen_suffix)
print("Dataset words supporting this choice:",
      suffix_counts.get(chosen_suffix, []))

# A single suffix is used to keep the RegexpStemmer rule interpretable.
regexp_stemmer = RegexpStemmer(chosen_suffix + "$")

def to_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

# POS tags are obtained in context from the ticket sentences.
contextual_tags = {}

for item in ticket_details:
    sentence_tokens = [
        str(token).lower()
        for token in item["Final Tokens"]
        if str(token).isalpha()
    ]

    if sentence_tokens:
        for word, tag in pos_tag(sentence_tokens):
            contextual_tags.setdefault(word, tag)

rows = []

for word in unique_final_tokens:
    tag = contextual_tags.get(word, "NN")
    wn_pos = to_wordnet_pos(tag)

    results = {
        "Word": word,
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp": regexp_stemmer.stem(word),
        "POS Lemma": lemmatizer.lemmatize(word, pos=wn_pos)
    }

    output_values = list(results.values())[1:]

    if len(set(output_values)) > 1:
        rows.append(results)

comparison_df = pd.DataFrame(
    rows,
    columns=["Word", "Porter", "Lancaster", "Regexp", "POS Lemma"]
)

display(comparison_df)

Q5

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from nltk.stem import PorterStemmer
from nltk.util import ngrams
import string
import math

# Build corpus sets from the generated tickets.
S1 = [
    token
    for item in ticket_details
    for token in item["Actual Tokens"]
]

S2 = [
    token for token in S1
    if not all(char in string.punctuation for char in token)
]

S3 = [
    str(token).lower()
    for item in ticket_details
    for token in item["Final Tokens"]
]

porter = PorterStemmer()
S4 = [porter.stem(token) for token in S3]

corpora = {
    "S1 - All tokens": S1,
    "S2 - Without punctuation": S2,
    "S3 - Final tokens": S3,
    "S4 - Porter stemmed": S4
}

# Total tokens, unique tokens and TTR
statistics = []

for name, corpus in corpora.items():
    total = len(corpus)
    unique = len(set(corpus))
    ttr = unique / total if total else 0

    statistics.append({
        "Corpus": name,
        "Total Tokens": total,
        "Unique Tokens": unique,
        "TTR": round(ttr, 4)
    })

stats_df = pd.DataFrame(statistics)
display(stats_df)

# Top 10 words of S1 and S3
print("\nTop 10 words in S1:")
print(Counter(S1).most_common(10))

print("\nTop 10 words in S3:")
print(Counter(S3).most_common(10))

# Log(rank) vs log(frequency) for S3
frequencies = Counter(S3)
ranked_frequencies = sorted(
    frequencies.values(), reverse=True
)

ranks = list(range(1, len(ranked_frequencies) + 1))

log_ranks = [math.log(rank) for rank in ranks]
log_frequencies = [
    math.log(freq) for freq in ranked_frequencies
]

plt.figure(figsize=(8, 5))
plt.plot(log_ranks, log_frequencies, marker=".", linestyle="-")
plt.xlabel("log(Rank)")
plt.ylabel("log(Frequency)")
plt.title("Log(Rank) vs Log(Frequency) for S3")
plt.grid(True)
plt.tight_layout()
plt.show()

# Top 5 bigrams of S1 and S3
print("\nTop 5 bigrams in S1:")
print(Counter(ngrams(S1, 2)).most_common(5))

print("\nTop 5 bigrams in S3:")
print(Counter(ngrams(S3, 2)).most_common(5))

Q6

In [1]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# (a) Compare TTR between S1 and S4
print("(a) TTR interpretation")
print(stats_df[["Corpus", "Total Tokens", "Unique Tokens", "TTR"]])

ttr_s1 = len(set(S1)) / len(S1) if S1 else 0
ttr_s4 = len(set(S4)) / len(S4) if S4 else 0

if ttr_s4 < ttr_s1:
    print("TTR decreased because normalization and stemming merged word forms.")
elif ttr_s4 > ttr_s1:
    print("TTR increased; compare token counts and normalization effects.")
else:
    print("TTR remained unchanged in this dataset.")

# (b) Find a real difference between the two tokenizers
print("\n(b) Tokenizer difference")

found = False

for item in ticket_details:
    text = get_raw_text(raw_tickets[item["Ticket"] - 1])
    custom = my_tokenize(text)
    standard = word_tokenize(text)

    if custom != standard:
        print("Example:", text)
        print("Custom tokenizer:", custom)
        print("NLTK tokenizer:", standard)
        found = True
        break

if not found:
    print("No difference found in these tickets; test the supplied TEST_LINES.")

# (c) Find an example of stemming changing a word undesirably
print("\n(c) Potentially undesirable stemming")

for word in unique_final_tokens:
    stem = porter.stem(word)
    lemma = lemmatizer.lemmatize(word)

    if stem != word and stem != lemma:
        print(f"Original: {word} | Porter stem: {stem} | "
              f"WordNet lemma: {lemma}")
        break

# (d) Find a verb for which POS-aware lemmatization differs from stemming
print("\n(d) POS-aware lemmatization")

found = False

for item in ticket_details:
    words = [
        str(token).lower()
        for token in item["Final Tokens"]
        if str(token).isalpha()
    ]

    for word, tag in pos_tag(words):
        if tag.startswith("V"):
            lemma = lemmatizer.lemmatize(word, pos=wordnet.VERB)
            stem = porter.stem(word)

            if lemma != stem:
                print(f"Word: {word}")
                print(f"POS tag: {tag}")
                print(f"Porter stem: {stem}")
                print(f"POS-aware lemma: {lemma}")
                found = True
                break

    if found:
        break

if not found:
    print("No suitable verb found; inspect the POS-tagged ticket sentences.")

(a) TTR interpretation


NameError: name 'stats_df' is not defined